# Notebook 4: Opening the Black Box. Filters, Feature Maps and Saliency

**Goal.** Look inside trained CNNs to see *what* they learn and *where* they look when they make a decision.

**What you will learn**

1. Convolution as feature detection, using classical hand-designed kernels.
2. Visualising learned first-layer filters (and why only the first layer is directly interpretable).
3. Capturing intermediate **feature maps** with forward hooks, and the feature hierarchy from edges to objects.
4. **Saliency maps**: gradients of the class score with respect to the input pixels.
5. **Grad-CAM**: class-discriminative localisation from the last conv layer.
6. **Activation maximisation**: synthesising an image that a given channel loves.
7. The limitations and pitfalls of all these methods.

We use a ResNet-18 pretrained on ImageNet (downloaded automatically) plus a small CNN we train from scratch on CIFAR-10. A GPU helps but is not required.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from skimage import data as skdata   # preinstalled in Colab; sample images

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Convolution as feature detection: hand-crafted kernels

Before deep learning, computer vision relied on hand-designed filters. Recall that a conv output is large where the image patch **correlates** with the kernel. Some classics:

* **Box blur**: $\frac{1}{9}\mathbf{1}_{3\times3}$. Averages neighbours, removes high frequencies.
* **Sobel** $G_x = \begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{pmatrix}$, $G_y = G_x^\top$. Approximate horizontal and vertical derivatives, so they respond to vertical and horizontal **edges**. Edge magnitude: $\sqrt{G_x^2 + G_y^2}$.
* **Laplacian** $\begin{pmatrix}0&1&0\\1&-4&1\\0&1&0\end{pmatrix}$. Second derivative, responds to blobs and fine detail in all directions.
* **Sharpen**: identity plus a negative Laplacian, boosting edges.

The point of deep learning is that the network discovers such kernels (and far more complex ones) by itself, because they are useful for minimising the loss.

In [ ]:
img_np = skdata.camera().astype(np.float32) / 255.0     # 512x512 grayscale
img = torch.from_numpy(img_np)[None, None]                # (1, 1, H, W)

kernels = {
    "box blur": torch.ones(3, 3) / 9,
    "sobel x": torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]),
    "sobel y": torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]),
    "laplacian": torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]),
    "sharpen": torch.tensor([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]]),
}
# Stack all kernels into ONE conv weight of shape (C_out=5, C_in=1, 3, 3): one call does all five
weight = torch.stack(list(kernels.values()))[:, None]
out = F.conv2d(img, weight, padding=1)[0]
edges = torch.sqrt(out[1] ** 2 + out[2] ** 2)

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
axes[0].imshow(img_np, cmap="gray"); axes[0].set_title("input")
for ax, (name, o) in zip(axes[1:], zip(kernels, out)):
    ax.imshow(o, cmap="gray"); ax.set_title(name)
axes[6].imshow(edges, cmap="gray"); axes[6].set_title("edge magnitude")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

Note how we computed 5 filtered images with **one** `conv2d` call by stacking kernels along the output-channel dimension. That is exactly what a `Conv2d(1, 5, 3)` layer does; in a CNN the 5 kernels are just learned rather than chosen.

## 2. A pretrained network

ResNet-18 was trained on ImageNet: 1.28M images, 1,000 classes. Its weights are a strong general-purpose visual feature extractor (Notebook 5 exploits this).

ImageNet models expect RGB images normalised with the ImageNet statistics, usually at $224\times224$. The `weights.transforms()` object gives exactly the preprocessing that was used during training.

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
resnet = models.resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def to_input(np_img):
    pil = transforms.functional.to_pil_image(np_img)
    return preprocess(pil).unsqueeze(0).to(device)

def denorm(t):
    # undo ImageNet normalisation for display; t is (3, H, W)
    return (t.cpu() * IMNET_STD + IMNET_MEAN).clamp(0, 1).permute(1, 2, 0)

images = {"cat": skdata.chelsea(), "coffee": skdata.coffee(), "astronaut": skdata.astronaut()}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, im) in zip(axes, images.items()):
    x = to_input(im)
    with torch.no_grad():
        probs = F.softmax(resnet(x), 1)[0]
    top = probs.topk(3)
    ax.imshow(denorm(x[0])); ax.axis("off")
    ax.set_title("\n".join(f"{categories[i]}: {p:.2f}" for p, i in zip(top.values, top.indices)), fontsize=9)
plt.tight_layout(); plt.show()

## 3. Visualising learned filters

The first conv layer of ResNet-18 is `Conv2d(3, 64, kernel_size=7, stride=2)`: 64 kernels of shape $3\times7\times7$. Because they act directly on RGB pixels, we can display each one as a small colour image (after rescaling to $[0,1]$).

You should see two families, the same ones neuroscientists found in the primary visual cortex (V1) of mammals:

* **Oriented edge / Gabor-like filters** at many orientations and frequencies, mostly colourless (luminance).
* **Colour-opponent blobs** (red vs green, blue vs yellow).

Deeper filters cannot be shown this way: a layer-3 kernel has hundreds of input channels, each of which is itself an abstract feature, so the weights are not an image. For those layers we look at **activations** instead (Sections 4 to 7).

In [ ]:
def show_filters(weight, ncols=16, title=""):
    w = weight.detach().cpu()
    w = (w - w.amin(dim=(1, 2, 3), keepdim=True)) / (w.amax(dim=(1, 2, 3), keepdim=True) - w.amin(dim=(1, 2, 3), keepdim=True) + 1e-8)
    nrows = int(np.ceil(w.shape[0] / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 0.8, nrows * 0.8))
    for ax in axes.flat: ax.axis("off")
    for ax, k in zip(axes.flat, w):
        ax.imshow(k.permute(1, 2, 0) if k.shape[0] == 3 else k[0], cmap=None if k.shape[0] == 3 else "gray",
                  interpolation="nearest")
    fig.suptitle(title); plt.show()

show_filters(resnet.conv1.weight, title="ResNet-18 conv1: 64 learned 7x7 RGB filters")
show_filters(models.resnet18(weights=None).conv1.weight, title="Same layer at random initialisation")

The contrast with random initialisation is the clearest evidence that structure was *learned from data*.

### Filters of a small CNN trained from scratch

Let's check that even a tiny network trained for a few minutes on CIFAR-10 develops edge and colour detectors. We give it a $5\times5$ first layer so the filters are big enough to see.

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
tf_train = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
cifar_train = datasets.CIFAR10("./data", train=True, download=True, transform=tf_train)
cifar_test = datasets.CIFAR10("./data", train=False, download=True, transform=tf_test)
train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(cifar_test, batch_size=512, num_workers=2)

small = nn.Sequential(
    nn.Conv2d(3, 32, 5, padding=2), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10),
).to(device)

show_filters(small[0].weight, title="Small CNN conv1 before training")

EPOCHS = 6
opt = torch.optim.Adam(small.parameters(), lr=2e-3)
for ep in range(EPOCHS):
    small.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); F.cross_entropy(small(x), y).backward(); opt.step()
    small.eval(); correct = n = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (small(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    print(f"epoch {ep + 1}: test acc {correct / n:.3f}")

show_filters(small[0].weight, title="Small CNN conv1 after training")

## 4. Feature maps: the hierarchy of representations

A **feature map** (or activation map) is the output of one channel of one layer: a 2D grid saying "how much of feature $c$ is present at each location". We capture them with **forward hooks** (introduced in Notebook 2), without touching the model code.

ResNet-18 is organised in stages:

| Stage | Output (for 224 input) | Typical content |
|---|---|---|
| `conv1` + `maxpool` | $64\times56\times56$ | edges, colours |
| `layer1` | $64\times56\times56$ | textures, corners |
| `layer2` | $128\times28\times28$ | repeated patterns, simple parts |
| `layer3` | $256\times14\times14$ | object parts (eyes, wheels, fur) |
| `layer4` | $512\times7\times7$ | whole objects, class-specific evidence |

As we go deeper: spatial resolution drops, the number of channels grows, receptive fields grow, and features become more **abstract**, **sparse** and **class-specific** (Zeiler and Fergus, 2014).

In [ ]:
activations = {}
def save_activation(name):
    def hook(module, inp, out):
        activations[name] = out.detach().cpu()
    return hook

layer_names = ["conv1", "layer1", "layer2", "layer3", "layer4"]
handles = [getattr(resnet, n).register_forward_hook(save_activation(n)) for n in layer_names]

x_cat = to_input(images["cat"])
with torch.no_grad():
    resnet(x_cat)
for h in handles: h.remove()

for n in layer_names:
    print(f"{n:<8}{tuple(activations[n].shape)}")

In [ ]:
def show_feature_maps(act, n=8, title=""):
    a = act[0]
    # pick the n most strongly activated channels (by mean activation)
    idx = a.mean(dim=(1, 2)).argsort(descending=True)[:n]
    fig, axes = plt.subplots(1, n, figsize=(n * 1.8, 2))
    for ax, i in zip(axes, idx):
        ax.imshow(a[i], cmap="viridis"); ax.set_title(f"ch {i.item()}", fontsize=8); ax.axis("off")
    fig.suptitle(title, y=1.05); plt.show()

plt.figure(figsize=(3, 3)); plt.imshow(denorm(x_cat[0])); plt.axis("off"); plt.title("input"); plt.show()
for n in layer_names:
    show_feature_maps(activations[n], title=f"{n}: {activations[n].shape[1]} channels, "
                                            f"{activations[n].shape[2]}x{activations[n].shape[3]}")

In [ ]:
# Sparsity: what fraction of units are exactly zero after ReLU, per stage?
# (conv1 is captured before its ReLU, so we apply one for a fair comparison)
for n in layer_names:
    a = F.relu(activations[n])
    print(f"{n:<8} fraction of zero activations: {(a == 0).float().mean():.2f}")

Early maps still look like the cat: they are filtered versions of the image. Deep maps are coarse ($7\times7$) blobs that fire on specific regions (often the face) and are mostly zero elsewhere.

## 5. Saliency maps (vanilla gradients)

**Question:** which pixels most affect the score of the predicted class?

Let $S_c(x)$ be the logit for class $c$. A first-order Taylor expansion around the image $x_0$ gives

$$
S_c(x) \approx S_c(x_0) + \nabla_x S_c(x_0)^\top (x - x_0).
$$

So the magnitude of $\nabla_x S_c$ at each pixel says how sensitive the score is to that pixel. Simonyan et al. (2014) take the max of $|\nabla_x S_c|$ over the colour channels to get a **saliency map**.

Implementation: set `requires_grad=True` on the **input**, run forward, call `backward()` on the chosen logit, read `x.grad`. This is the same backpropagation used for training, just differentiating with respect to the input rather than the weights.

**SmoothGrad** (Smilkov et al., 2017) reduces the notorious noise of raw gradients by averaging saliency over several noisy copies of the input.

In [ ]:
def saliency(model, x, target=None, smooth_n=0, sigma=0.15):
    model.eval()
    xs = [x] if smooth_n == 0 else [x + sigma * torch.randn_like(x) for _ in range(smooth_n)]
    grads = 0
    for xi in xs:
        xi = xi.clone().requires_grad_(True)
        logits = model(xi)
        c = logits.argmax(1).item() if target is None else target
        model.zero_grad()
        logits[0, c].backward()
        grads = grads + xi.grad.abs()
    return (grads[0].amax(0) / len(xs)).cpu(), c

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, (name, im) in zip(axes, images.items()):
    x = to_input(im)
    s_vanilla, c = saliency(resnet, x)
    s_smooth, _ = saliency(resnet, x, target=c, smooth_n=25)
    row[0].imshow(denorm(x[0])); row[0].set_title(categories[c])
    row[1].imshow(s_vanilla, cmap="hot"); row[1].set_title("vanilla gradient")
    row[2].imshow(s_smooth, cmap="hot"); row[2].set_title("SmoothGrad (n=25)")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

## 6. Grad-CAM

Saliency maps are high-resolution but noisy and often not very class-specific. **Grad-CAM** (Selvaraju et al., 2017) works at the last convolutional layer instead, whose units have large receptive fields and encode high-level concepts while still keeping (coarse) spatial layout.

Let $A^k \in \mathbb{R}^{H\times W}$ be the $k$-th feature map of the last conv stage (`layer4`, $7\times7$) and $y^c$ the logit of class $c$.

1. **Channel importance** = global-average-pooled gradient:
$$
\alpha_k^c = \frac{1}{HW}\sum_{i,j}\frac{\partial y^c}{\partial A^k_{ij}}.
$$
2. **Weighted combination**, keeping only positive evidence:
$$
L^c_\text{Grad-CAM} = \text{ReLU}\Big(\sum_k \alpha_k^c A^k\Big).
$$
3. Upsample the $7\times7$ map to the image size and overlay.

**Why it makes sense:** for networks that end in global average pooling plus one linear layer (like ResNet), $\alpha_k^c$ is exactly proportional to the linear weight $w_k^c$ connecting channel $k$ to class $c$, and Grad-CAM reduces to the earlier **Class Activation Map** (CAM, Zhou et al., 2016): "sum the feature maps, weighted by how much each channel votes for class $c$".

We need both the activations *and* their gradients, so we register a forward hook that also attaches a tensor hook to capture the gradient during `backward()`.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        out.requires_grad_(True)                                        # ensure we can hook
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x, target=None):
        self.model.eval()
        logits = self.model(x)
        c = logits.argmax(1).item() if target is None else target
        self.model.zero_grad()
        logits[0, c].backward()
        alpha = self.grads.mean(dim=(2, 3), keepdim=True)              # (1, K, 1, 1)
        cam = F.relu((alpha * self.acts).sum(1, keepdim=True))          # (1, 1, h, w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam[0, 0].detach().cpu(), c

    def remove(self):
        self.handle.remove()

gradcam = GradCAM(resnet, resnet.layer4)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (name, im) in zip(axes, images.items()):
    x = to_input(im)
    cam, c = gradcam(x)
    ax.imshow(denorm(x[0])); ax.imshow(cam, cmap="jet", alpha=0.45)
    ax.set_title(f"Grad-CAM: {categories[c]}"); ax.axis("off")
plt.tight_layout(); plt.show()

### Grad-CAM is class-discriminative

The same image can be explained for *different* classes. Below, we ask "where is the evidence for each of the top-3 classes?" for the astronaut photo.

In [ ]:
x = to_input(images["astronaut"])
with torch.no_grad():
    top3 = resnet(x).topk(3).indices[0].tolist()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, c in zip(axes, top3):
    cam, _ = gradcam(x, target=c)
    ax.imshow(denorm(x[0])); ax.imshow(cam, cmap="jet", alpha=0.45)
    ax.set_title(categories[c]); ax.axis("off")
plt.tight_layout(); plt.show()
gradcam.remove()

## 7. Activation maximisation: what does a channel want to see?

Instead of asking which part of a real image activates a unit, we can **synthesise** the image that maximises it. Starting from noise, do gradient **ascent** on the input:

$$
x^* = \arg\max_x \; \overline{A^k_\ell(x)} \; - \; \lambda \, \mathcal{R}(x),
$$

where $\overline{A^k_\ell}$ is the mean activation of channel $k$ in layer $\ell$ and $\mathcal{R}$ is a regulariser. Without regularisation the result is high-frequency noise (an adversarial pattern), so we use the standard tricks from Olah et al. (2017), "Feature Visualization": small random **jitter** each step (robustness to translation), a total-variation penalty (smoothness) and an L2 penalty.

Note we optimise the **image**, not the weights: the model is frozen.

In [ ]:
def total_variation(x):
    return (x[..., 1:, :] - x[..., :-1, :]).abs().mean() + (x[..., :, 1:] - x[..., :, :-1]).abs().mean()

def visualise_channel(model, layer, channel, steps=200, size=128, lr=0.05, tv=0.5, l2=1e-3):
    for p in model.parameters(): p.requires_grad_(False)
    store = {}
    h = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    x = (0.1 * torch.randn(1, 3, size, size, device=device)).requires_grad_(True)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(steps):
        dx, dy = np.random.randint(-8, 9, size=2)
        model(torch.roll(x, shifts=(int(dx), int(dy)), dims=(2, 3)))
        loss = -store["a"][0, channel].mean() + tv * total_variation(x) + l2 * (x ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    h.remove()
    for p in model.parameters(): p.requires_grad_(True)
    return denorm(x.detach()[0])

STEPS = 200
targets = [(resnet.layer2, 5), (resnet.layer2, 40), (resnet.layer3, 10), (resnet.layer3, 100),
           (resnet.layer4, 50), (resnet.layer4, 300)]
names = ["layer2", "layer2", "layer3", "layer3", "layer4", "layer4"]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.3))
for ax, (layer, ch), nm in zip(axes, targets, names):
    ax.imshow(visualise_channel(resnet, layer, ch, steps=STEPS)); ax.set_title(f"{nm}, ch {ch}"); ax.axis("off")
plt.tight_layout(); plt.show()

Mid-level channels typically produce repeating textures and patterns; deeper channels produce more object-like fragments (eyes, fur, wheels), though they become harder to read without stronger regularisation (frequency-domain parametrisation, transformation robustness, as in the Lucid library).

## 8. Caveats: how much should we trust these pictures?

* **Saliency maps can be misleading.** Adebayo et al. (2018), "Sanity Checks for Saliency Maps", showed that some methods produce nearly identical maps for a trained network and one with *randomised weights*: they behave like edge detectors on the input rather than explanations of the model. Vanilla gradients and Grad-CAM pass these checks reasonably; guided backpropagation does not. Exercise 4 lets you check this yourself.
* **Grad-CAM is coarse** (7x7 for ResNet), so it localises regions, not precise boundaries.
* **Explanations are correlational.** A highlighted region tells you what the output is sensitive to, not *why* in any causal sense. A model relying on a watermark or background will show it, which is exactly why these tools are useful for debugging shortcuts.
* **Single channels are not always single concepts.** Many units are *polysemantic*, responding to several unrelated things.

## Summary

* Convolution with a kernel = template matching; classical kernels detect edges, blobs and blur.
* First-layer CNN filters converge to Gabor-like edge and colour-opponent detectors; deeper layers must be studied via activations.
* Forward hooks capture feature maps; deeper features are lower-resolution, higher-level, sparser.
* **Saliency** $= |\nabla_x S_c|$; **SmoothGrad** averages over noisy inputs.
* **Grad-CAM** $= \text{ReLU}(\sum_k \alpha_k^c A^k)$ with $\alpha_k^c$ the pooled gradients; class-discriminative and coarse.
* **Activation maximisation** optimises the input to excite a unit; needs regularisation.

## Exercises

1. Design a $5\times5$ kernel that detects diagonal ($45°$) edges and apply it to the camera image.
2. Run Grad-CAM on `layer3` instead of `layer4`. What do you gain and lose?
3. Apply Grad-CAM to the small CIFAR model (`small[8]`, the last conv). Look at misclassified test images: where was the network looking?
4. **Sanity check.** Compute the saliency map and Grad-CAM with `models.resnet18(weights=None)` (random weights). Do the maps still look meaningful? What does this tell you?
5. **Occlusion sensitivity** (a model-agnostic alternative): slide a grey $32\times32$ patch over the image, record the drop in the target class probability at each position, and plot the resulting heatmap. Compare it with Grad-CAM.
6. Use `visualise_channel` with the final `fc` layer as the hook target and a class index (for example 281, "tabby cat") to synthesise a class visualisation.